# Deflation-PINNs: Learning Multiple Solutions for PDEs and Landau-de Gennes

**Paper:** Disaro, S., Maity, R.R., Bacho, A. (2026). *Deflation-PINNs: Learning Multiple Solutions for PDEs and Landau-de Gennes.* arXiv:2603.27936 [math.NA].

**Carpeta origen:** `PINNs/4. Otros/Deflation-PINNs_Learning_Multiple_Solutions_for_PD.pdf`

## Como se usan las PINNs en este paper

Una PINN estandar solo encuentra **una** solucion a la vez, aunque el problema no lineal tenga varias soluciones distintas (p.ej. el modelo de Landau-de Gennes de cristales liquidos tiene 6 configuraciones de equilibrio estables en un dominio cuadrado). El paper propone **Deflation-PINNs**, que aprenden **$K$ soluciones simultaneamente** con dos ingredientes clave (Fig. 1):

1. **Arquitectura tipo DeepONet simplificada**: una red troncal compartida $\tau:\mathbb{R}^d\to\mathbb{R}^p$ produce $p$ funciones base $\tau_i(x)$, y cada una de las $K$ soluciones se representa como una combinacion lineal **con sus propios pesos de rama** $\beta^k\in\mathbb{R}^p$ (parametros libres, no una red): $\tilde G(k,x)=\sum_{i=1}^p \tau_i(x)\beta_i^k\approx u_k(x)$. Todas las soluciones comparten la misma base $\tau$, pero cada una tiene su propia combinacion.
2. **Perdida de deflacion**: ademas de la perdida fisica estandar (residuo de la EDP para cada una de las $K$ soluciones), se anade un termino que **penaliza que dos soluciones aprendidas esten demasiado cerca** entre si:

$$\mathcal{L}_{Def}(\tilde G) = \frac{2}{K(K-1)}\sum_{i=1}^K\sum_{j=i+1}^K \max\Big(1-\frac{1}{d_{min}}\|\tilde G(i,\cdot)-\tilde G(j,\cdot)\|_{L^2},\,0\Big)$$

$$\mathcal{L}_{total}(\tilde G)=\alpha\sum_{k=1}^K \mathcal{E}_G(\tilde G(k,\cdot)) + \beta\cdot\mathcal{L}_{Def}(\tilde G)$$

Esto fuerza a las $K$ ramas a converger a soluciones **distintas** (separadas al menos $d_{min}$ en norma $L^2$), en vez de colapsar todas a la misma solucion (el riesgo natural al minimizar solo la perdida fisica).

Este cuaderno reproduce fielmente el mecanismo de Deflation-PINNs (arquitectura de tronco compartido + pesos de rama + perdida de deflacion) sobre un **problema 1D de tipo Ginzburg-Landau/pozo doble** con **tres soluciones constantes conocidas** ($u=-1,0,1$), un analogo simplificado en 1D del modelo completo de Landau-de Gennes 2D del paper (que usa un campo tensorial $\mathbf{Q}$ y 6 soluciones): $-u''+\frac{2}{\varepsilon^2}(u^3-u)=0$ en $[0,1]$, con condicion de Neumann homogenea $u'(0)=u'(1)=0$ (que satisfacen exactamente las tres soluciones constantes).

## Repositorio publico

El paper **incluye explicitamente** su repositorio de codigo en el texto (Seccion 3): "The code can be found in https://github.com/SeanDisaro/DeflationPINNs".

- **SeanDisaro/DeflationPINNs** &mdash; https://github.com/SeanDisaro/DeflationPINNs

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Problema 1D de pozo doble (analogo simplificado de Landau-de Gennes, Eq. 3.2): $-u''+\frac{2}{\varepsilon^2}(u^3-u)=0$, Neumann homogenea, 3 soluciones constantes conocidas: $u=-1,0,1$

In [ ]:
eps = 0.5
K = 3       # numero de soluciones a aprender simultaneamente
p = 16      # dimension de la representacion (funciones base del tronco)
d_min = 0.5  # distancia minima deseada entre soluciones (hiperparametro, Seccion 3.2)

N_col = 100
x_col = torch.linspace(0, 1, N_col, device=device).view(-1, 1).requires_grad_(True)


def d_dx(f, x):
    return torch.autograd.grad(f, x, grad_outputs=torch.ones_like(f),
                                create_graph=True, retain_graph=True)[0]

## 2. Arquitectura Deflation-PINN (Fig. 1): tronco compartido $\tau(x)$ + $K$ pesos de rama $\beta^k$

In [ ]:
class Trunk(nn.Module):
    """tau: R -> R^p, red compartida por todas las K soluciones (Fig. 1)."""
    def __init__(self, p=p, n_hidden=3, n_neurons=64):
        super().__init__()
        layers = [nn.Linear(1, n_neurons), nn.Tanh()]
        for _ in range(n_hidden - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.Tanh()]
        layers += [nn.Linear(n_neurons, p)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)  # (N, p)


trunk = Trunk().to(device)
branch_weights = nn.Parameter(torch.randn(K, p, device=device) * 0.5)  # beta^k, k=1..K


def G(x):
    """Devuelve (N, K): las K soluciones evaluadas en x, todas comparten tau(x)."""
    tau = trunk(x)                  # (N, p)
    return tau @ branch_weights.T    # (N, K)

## 3. Perdida fisica (residuo, por cada una de las K ramas) + perdida de deflacion

In [ ]:
def compute_loss(alpha=1.0, beta_def=2.0):
    U = G(x_col)  # (N, K)

    loss_phys = 0.0
    for k in range(K):
        uk = U[:, k:k + 1]
        duk = d_dx(uk, x_col)
        d2uk = d_dx(duk, x_col)
        residual = -d2uk + (2 / eps**2) * (uk**3 - uk)
        loss_phys = loss_phys + torch.mean(residual**2)
        # condicion de Neumann homogenea u'(0)=u'(1)=0 (perdida blanda)
        loss_phys = loss_phys + 10.0 * (duk[0]**2 + duk[-1]**2).squeeze()

    # Perdida de deflacion: fuerza a las K soluciones a mantenerse separadas (formula del paper)
    loss_def = 0.0
    n_pairs = 0
    for i in range(K):
        for j in range(i + 1, K):
            dist_l2 = torch.sqrt(torch.mean((U[:, i] - U[:, j])**2) + 1e-12)
            loss_def = loss_def + torch.clamp(1 - dist_l2 / d_min, min=0.0)
            n_pairs += 1
    loss_def = loss_def / n_pairs

    return alpha * loss_phys + beta_def * loss_def, loss_phys.item(), loss_def.item()

## 4. Entrenamiento

In [ ]:
optimizer = torch.optim.Adam(list(trunk.parameters()) + [branch_weights], lr=1e-3)
history = []
for epoch in range(6000):
    optimizer.zero_grad()
    loss, l_phys, l_def = compute_loss()
    loss.backward()
    optimizer.step()
    history.append(loss.item())
    if epoch % 1000 == 0:
        print(f'epoch {epoch:5d} | loss={loss.item():.4e} | phys={l_phys:.4e} | deflation={l_def:.4e}')

## 5. Resultados: las K=3 ramas deben converger a las 3 soluciones distintas ($u=-1,0,1$), cf. Fig. 2 del paper

In [ ]:
x_plot = torch.linspace(0, 1, 200, device=device).view(-1, 1)
with torch.no_grad():
    U_plot = G(x_plot).cpu().numpy()
x_plot_np = x_plot.cpu().numpy().flatten()

plt.figure(figsize=(7, 4.5))
for k in range(K):
    plt.plot(x_plot_np, U_plot[:, k], label=f'rama k={k+1} (media={U_plot[:,k].mean():.2f})')
for level in [-1, 0, 1]:
    plt.axhline(level, color='gray', linestyle=':', alpha=0.5)
plt.xlabel('x'); plt.ylabel('u_k(x)')
plt.title('Deflation-PINN: 3 soluciones aprendidas simultaneamente')
plt.legend()
plt.show()

plt.figure(figsize=(6, 4))
plt.semilogy(history)
plt.xlabel('Epoca'); plt.ylabel('Loss total (escala log)')
plt.title('Convergencia')
plt.show()

print('Valores medios de cada rama (deberian aproximarse a -1, 0 y 1, en algun orden):')
print(U_plot.mean(axis=0))